In [4]:
""" 01_step_responses.png      theta(t) of every level, both wheels
    02_step_response_fit.png   measured theta/omega vs the odeint simulation (--pwm)
    03_omega_vs_U.png          omega_ss(U), both wheels, k_e and dead zone marked
    04_Tm_vs_pwm.png           time constant against PWM
    05_fit_grid_wheel_<N>.png  every level: measurement vs fitted model
    06_model_vs_measurement.png one-number model (k_e, median T_m) vs data
    07_wheel_mismatch.png      right/left speed ratio against PWM
    08_motor_graphs.png        per wheel: omega vs PWM % (linear fit) and U/omega
    09_omega_all_wheels.png    omega(U) for all wheels on one axes
"""

import argparse
import os
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm  # noqa: E402

from identify import load  # noqa: E402
from motor_lib import (BATTERY_V, COLORS, GRID, INK, INK_2, LINEAR_FIT_MIN_PWM,  # noqa: E402
                       MUTED, WHEELS, measured_omega, omega_model, read_raw_log,
                       simulate, style_axes, wheel_label)

HERE = os.path.dirname(os.path.abspath('mtr_idf.txt'))
MARKERS = {1: "o", 2: "s"}
# Diverging PWM scale: reverse = violet, 0 = neutral grey, forward = green.
PWM_CMAP = LinearSegmentedColormap.from_list(
    "pwm", ["#4a3bb8", "#9a92e0", "#d9d8d4", "#5fb88f", "#0d7a50"])

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "savefig.facecolor": "#fcfcfb",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "legend.frameon": False,
    "legend.fontsize": 9,
})


def save(fig, out, name):
    path = os.path.join(out, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", os.path.relpath(path))


# figures

def fig_step_responses(runs, out):
    norm = TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
    for ax, w in zip(axs, (1, 2)):
        for pwm, r in sorted(runs[w].items()):
            ax.plot(r.t, r.theta, color=PWM_CMAP(norm(pwm)), lw=1.4)
        ax.set_title(wheel_label(w))
        ax.set_xlabel("t since step, s")
        style_axes(ax)
    axs[0].set_ylabel("θ(t), rad")
    sm = plt.cm.ScalarMappable(norm=norm, cmap=PWM_CMAP)
    cb = fig.colorbar(sm, ax=axs, pad=0.02, ticks=[-100, -50, 0, 50, 100])
    cb.set_label("PWM, %")
    cb.outline.set_visible(False)
    fig.suptitle("Step responses θ(t), all PWM levels (re-zeroed at the step)",
                 fontweight="bold")
    save(fig, out, "01_step_responses.png")


def fig_step_fit(runs, fits, models, pwm, out):
    """Required plot: measured theta(t) and the simulated model on one axis."""
    fig, axs = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True)
    for j, w in enumerate((1, 2)):
        r = runs[w][pwm]
        f = next(x for x in fits[w] if x.pwm == pwm)
        m = models[w]
        t_dense = np.linspace(0, r.t[-1], 400)
        # (a) identified for this level: k_e = U / w_ss, T_m from curve_fit
        w_a, th_a = simulate(t_dense, r.U, f.T_m, r.U / f.w_ss)
        # (b) one-number wheel model: k_e through origin, median T_m
        w_b, th_b = simulate(t_dense, r.U, m.T_m_median, m.k_e)

        ax = axs[0, j]
        ax.plot(r.t, r.theta, MARKERS[w], color=COLORS[w], ms=5, mec="#fcfcfb",
                mew=0.8, label="measured", zorder=3)
        ax.plot(t_dense, th_a, color=INK, lw=2,
                label=f"odeint, this level: T_m={f.T_m:.3f} s, ω_ss={f.w_ss:.2f} rad/s")
        ax.plot(t_dense, th_b, color=INK_2, lw=1.5, ls="--",
                label=f"odeint, wheel model: T_m={m.T_m_median:.3f} s, k_e={m.k_e:.3f}")
        ax.set_title(f"{wheel_label(w)} — PWM {pwm:+d} % (U = {r.U:.2f} V)")
        ax.set_ylabel("θ, rad")
        ax.legend(loc="upper left" if pwm > 0 else "lower left")
        res = np.degrees(r.theta - np.interp(r.t, t_dense, th_a))
        ax.text(0.98, 0.04 if pwm > 0 else 0.92,
                f"RMSE {np.sqrt(np.mean(res ** 2)):.1f}°", transform=ax.transAxes,
                ha="right", color=INK_2, fontsize=9)

        ax = axs[1, j]
        tm, wm = measured_omega(r)
        ax.plot(tm, wm, MARKERS[w], color=COLORS[w], ms=5, mec="#fcfcfb", mew=0.8,
                label="measured Δθ/Δt", zorder=3)
        ax.plot(t_dense, w_a, color=INK, lw=2, label="odeint, this level")
        ax.plot(t_dense, w_b, color=INK_2, lw=1.5, ls="--", label="odeint, wheel model")
        ax.axhline(f.w_ss, color=MUTED, lw=1, ls=":")
        ax.set_ylabel("ω, rad/s")
        ax.set_xlabel("t since step, s")
        ax.legend(loc="lower right" if pwm > 0 else "upper right")
    for ax in axs.flat:
        style_axes(ax)
    fig.suptitle("Step response: measurement vs simulated first-order motor",
                 fontweight="bold")
    fig.tight_layout()
    save(fig, out, "02_step_response_fit.png")


def _omega_U_axes(ax, fits, models, U_lim=None):
    for w in (1, 2):
        mv = [f for f in fits[w] if f.moved]
        st = [f for f in fits[w] if not f.moved]
        U = np.array([f.U for f in mv])
        ax.plot(U, [f.w_ss for f in mv], MARKERS[w], color=COLORS[w], ms=6,
                mec="#fcfcfb", mew=0.8, zorder=4, label=f"{wheel_label(w)}")
        ax.plot([f.U for f in st], [0] * len(st), MARKERS[w], ms=6, mfc="none",
                mec=COLORS[w], zorder=4)
        m = models[w]
        Ul = np.linspace(-11.5, 11.5, 2)
        ax.plot(Ul, m.omega_ss_linear(Ul), color=COLORS[w], lw=1.5, ls="--",
                label=f"fit ω = U / k_e, k_e = {m.k_e:.3f} V·s/rad")
        for side in (m.pos, m.neg):
            sgn = 1 if side is m.pos else -1
            Ua = np.linspace(sgn * LINEAR_FIT_MIN_PWM / 100 * mv[0].U / mv[0].pwm * 100,
                             sgn * 11.2, 2)
            ax.plot(Ua, side(Ua), color=COLORS[w], lw=1, alpha=0.55)
    m1, m2 = models[1], models[2]
    lo = min(m1.dz_bound[0], m2.dz_bound[0])
    hi = max(m1.dz_bound[1], m2.dz_bound[1])
    ax.axvspan(lo, hi, color=MUTED, alpha=0.18, lw=0, zorder=1,
               label=f"dead zone lies inside ±{hi:.2f} V (0 % still, ±5 % moving)")
    for w in (1, 2):
        for e in models[w].dz_estimate:
            ax.axvline(e, color=COLORS[w], lw=1, ls=":", zorder=2)
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.axvline(0, color=MUTED, lw=0.8)
    if U_lim:
        ax.set_xlim(-U_lim, U_lim)
        ax.set_ylim(-U_lim * 3.2, U_lim * 3.2)
    ax.set_xlabel("U = PWM/100 · U_battery, V")
    ax.set_ylabel("ω_ss, rad/s")
    style_axes(ax)


def fig_omega_vs_U(fits, models, battery, out):
    """Required plot: omega_ss(U), both wheels, k_e and the dead zone."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6),
                                   gridspec_kw=dict(width_ratios=[1.6, 1]))
    _omega_U_axes(ax1, fits, models)
    ax1.set_title(f"ω_ss vs U, both wheels (U_battery = {battery:.2f} V)")
    handles, labels = ax1.get_legend_handles_labels()
    handles.append(plt.Line2D([], [], color=INK_2, lw=1, alpha=0.55))
    labels.append(f"affine fit for |PWM| ≥ {LINEAR_FIT_MIN_PWM} % (per direction)")
    handles.append(plt.Line2D([], [], color=INK_2, lw=1, ls=":"))
    labels.append("dead-zone edge estimate (two lowest levels)")
    ax1.legend(handles, labels, loc="upper left", fontsize=8.5)
    _omega_U_axes(ax2, fits, models, U_lim=2.4)
    ax2.set_title("zoom near zero — open markers: no motion")
    fig.tight_layout()
    save(fig, out, "03_omega_vs_U.png")


def fig_Tm(fits, out):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    for w, dx in ((1, -0.8), (2, 0.8)):
        mv = [f for f in fits[w] if f.moved]
        ax.errorbar([f.pwm + dx for f in mv], [f.T_m for f in mv],
                    yerr=[f.T_m_std for f in mv], fmt=MARKERS[w] + "-",
                    color=COLORS[w], ms=6, mec="#fcfcfb", mew=0.8, lw=1.2,
                    capsize=2, label=wheel_label(w))
    dt = np.median([f.dt_ms for f in fits[1]]) / 1000
    ax.axhspan(0, 2 * dt, color=MUTED, alpha=0.18, lw=0,
               label=f"< 2 sample periods ({2 * dt * 1000:.0f} ms): poorly resolved")
    ax.set_xlabel("PWM, %")
    ax.set_ylabel("T_m, s")
    ax.set_ylim(bottom=0)
    ax.set_xticks(range(-100, 101, 20))
    ax.set_title("Electromechanical time constant T_m against PWM")
    ax.legend(loc="upper left")
    style_axes(ax)
    save(fig, out, "04_Tm_vs_pwm.png")


def fig_fit_grid(runs, fits, out):
    for w in (1, 2):
        levels = sorted(runs[w])
        ncol = 7
        nrow = int(np.ceil(len(levels) / ncol))
        fig, axs = plt.subplots(nrow, ncol, figsize=(16, 2.1 * nrow))
        for ax, pwm in zip(axs.flat, levels):
            r = runs[w][pwm]
            f = next(x for x in fits[w] if x.pwm == pwm)
            ax.plot(r.t, np.degrees(r.theta), MARKERS[w], color=COLORS[w], ms=3)
            if f.moved:
                td = np.linspace(0, r.t[-1], 200)
                _, th = simulate(td, r.U, f.T_m, r.U / f.w_ss)
                ax.plot(td, np.degrees(th), color=INK, lw=1.2)
                txt = f"T_m {f.T_m:.2f} s\nω {f.w_ss:.1f} rad/s"
            else:
                txt = "no motion"
            ax.set_title(f"{pwm:+d} %", fontsize=9)
            ax.text(0.04 if pwm >= 0 else 0.96, 0.95, txt,
                    transform=ax.transAxes, fontsize=7, color=INK_2,
                    ha="left" if pwm >= 0 else "right", va="top")
            style_axes(ax)
            ax.tick_params(labelsize=7)
        for ax in list(axs.flat)[len(levels):]:
            ax.axis("off")
        fig.suptitle(f"{wheel_label(w)}: measured θ (deg, markers) vs identified "
                     f"model (line), x = t since step (s)", fontweight="bold")
        fig.tight_layout()
        save(fig, out, f"05_fit_grid_wheel_{w}.png")


def fig_model_vs_measurement(runs, models, out, levels=(-100, -60, -30, -10, 10, 30, 60, 100)):
    """How far the one-number linear model (k_e, median T_m) is from the data."""
    fig, axs = plt.subplots(2, 4, figsize=(15, 7), sharex=True)
    for ax, pwm in zip(axs.flat, levels):
        for w in (1, 2):
            r, m = runs[w][pwm], models[w]
            td = np.linspace(0, r.t[-1], 300)
            _, th = simulate(td, r.U, m.T_m_median, m.k_e)
            ax.plot(r.t, r.theta, MARKERS[w], color=COLORS[w], ms=3.5, alpha=0.9,
                    label=f"measured, wheel {w}")
            ax.plot(td, th, color=COLORS[w], lw=1.5, ls="--",
                    label=f"model, wheel {w}")
            err = 100 * (abs(th[-1]) / abs(r.theta[-1]) - 1)   # <0: model too slow
            ax.text(0.03 if pwm > 0 else 0.97, 0.88 - 0.08 * (w - 1),
                    f"wheel {w}: |θ| at 2 s {err:+.0f} %", transform=ax.transAxes,
                    fontsize=8, color=INK_2, ha="left" if pwm > 0 else "right")
        ax.set_title(f"PWM {pwm:+d} %")
        style_axes(ax)
    for ax in axs[1]:
        ax.set_xlabel("t since step, s")
    for ax in axs[:, 0]:
        ax.set_ylabel("θ, rad")
    axs[0, 0].legend(loc="lower left", fontsize=8)
    fig.suptitle("One-number model ω_ss = U/k_e, T_m = median — where the linear model fails",
                 fontweight="bold")
    fig.tight_layout()
    save(fig, out, "06_model_vs_measurement.png")


def fig_mismatch(fits, out):
    pairs = [(a, b) for a, b in zip(fits[1], fits[2]) if a.moved and b.moved]
    pwm = np.array([a.pwm for a, _ in pairs])
    diff = np.array([100 * (b.w_ss / a.w_ss - 1) for a, b in pairs])
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.axhline(0, color=MUTED, lw=1)
    for sgn in (-1, 1):
        k = np.sign(pwm) == sgn
        ax.plot(pwm[k], diff[k], "o-", color=INK_2, ms=6, lw=1.3, mec="#fcfcfb")
    for side, k in (("reverse", pwm <= -LINEAR_FIT_MIN_PWM),
                    ("forward", pwm >= LINEAR_FIT_MIN_PWM)):
        mean = diff[k].mean()
        x = pwm[k]
        ax.plot([x.min(), x.max()], [mean, mean], color=INK, lw=2)
        ax.text(x.mean(), mean + (2.2 if side == "reverse" else -4.5), f" {side} mean {mean:+.1f} %", ha="center",
                color=INK, fontsize=9,
                bbox=dict(fc="#fcfcfb", ec="none", pad=1.5))
    # Levels just above the dead zone have huge ratios; keep them readable.
    lim = 20
    for x, y in zip(pwm, diff):
        if abs(y) > lim:
            ax.annotate(f"{x:+d} %: {y:+.0f} % (off scale)",
                        xy=(x, np.sign(y) * lim), xytext=(x + 6, np.sign(y) * (lim - 4)),
                        color=INK_2, fontsize=8.5,
                        arrowprops=dict(arrowstyle="->", color=INK_2, lw=0.8))
    ax.set_ylim(-lim - 1, lim + 1)
    ax.set_xlabel("PWM, %")
    ax.set_ylabel("ω_ss,right / ω_ss,left − 1, %")
    ax.set_xticks(range(-100, 101, 20))
    ax.set_title("Wheel mismatch at equal PWM (positive: right wheel faster)")
    style_axes(ax)
    save(fig, out, "07_wheel_mismatch.png")


def fig_motor_graphs(fits, models, battery, out):
    """Course-reference layout (src/plot.py): per wheel, omega vs PWM % and U/omega."""
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    for row, w in zip(axs, (1, 2)):
        mv = [f for f in fits[w] if f.moved]
        pwm = np.array([f.pwm for f in fits[w]])
        w_ss = np.array([f.w_ss for f in fits[w]])

        # omega_ss vs PWM % with a straight-line fit over all levels
        ax = row[0]
        k, b = np.polyfit(pwm, w_ss, 1)
        x = np.linspace(-100, 100, 2)
        ax.plot(x, k * x + b, color=INK, lw=1.5, ls="--",
                label=f"linear fit: k = {k:.4f} rad/s per % "
                      f"({np.degrees(k):.2f} °/s per %)")
        ax.plot(pwm, w_ss, MARKERS[w], color=COLORS[w], ms=6, mec="#fcfcfb",
                mew=0.8, zorder=3, label="ω_ss, measured")
        ax.axhline(0, color=MUTED, lw=0.8)
        ax.axvline(0, color=MUTED, lw=0.8)
        ax.set_xlabel("PWM (U%), %")
        ax.set_ylabel("ω_ss, rad/s")
        ax.set_xticks(range(-100, 101, 20))
        ax.set_title(f"{wheel_label(w)} — ω vs U%")
        ax.legend(loc="upper left")
        style_axes(ax)

        # U / omega_ss for every moving level, with mean and fitted k_e
        ax = row[1]
        p = np.array([f.pwm for f in mv])
        ratio = np.array([f.U / f.w_ss for f in mv])
        ax.bar(p, ratio, width=3.6, color=COLORS[w], zorder=3)
        mean = ratio.mean()
        ax.axhline(mean, color=INK, lw=1.5, ls="--",
                   label=f"mean U/ω = {mean:.3f} V·s/rad")
        ax.axhline(models[w].k_e, color=INK_2, lw=1.2, ls=":",
                   label=f"k_e from fit ω = U/k_e = {models[w].k_e:.3f} V·s/rad")
        ax.set_xlabel("PWM (U%), %")
        ax.set_ylabel("U/ω_ss, V·s/rad")
        ax.set_xticks(range(-100, 101, 20))
        ax.set_ylim(0, ratio.max() * 1.25)
        ax.set_title(f"{wheel_label(w)} — coefficient U/ω (0 % excluded)")
        ax.legend(loc="upper center", ncol=1)
        style_axes(ax)
    fig.suptitle(f"Motor characteristics per wheel (U_battery = {battery:.2f} V)",
                 fontweight="bold")
    fig.tight_layout()
    save(fig, out, "08_motor_graphs.png")


def fig_omega_all_wheels(fits, battery, out):
    """Course-reference layout (src/plot_omega.py): omega(U) for all wheels."""
    fig, ax = plt.subplots(figsize=(12, 6))
    for w in (1, 2):
        U = [f.U for f in fits[w]]
        w_ss = [f.w_ss for f in fits[w]]
        ax.plot(U, w_ss, "-" + MARKERS[w], color=COLORS[w], lw=1.5, ms=6,
                mec="#fcfcfb", mew=0.8, label=wheel_label(w))
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.axvline(0, color=MUTED, lw=0.8)
    ax.set_xlabel("motor voltage U = PWM/100 · U_battery, V")
    ax.set_ylabel("ω_ss, rad/s")
    ax.set_title(f"All wheels — ω depending on U (U_battery = {battery:.2f} V)")
    ax.legend(loc="upper left")
    style_axes(ax)
    save(fig, out, "09_omega_all_wheels.png")


def main():
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--raw", default=os.path.join(HERE, "mtr_idf.txt"))
    ap.add_argument("--out", default=os.path.join(HERE, "figures"))
    ap.add_argument("--battery", type=float, default=BATTERY_V)
    ap.add_argument("--pwm", type=int, default=50,
                    help="PWM level for the detailed step-response figure")
    args, unknown = ap.parse_known_args()
    os.makedirs(args.out, exist_ok=True)

    raw = read_raw_log(args.raw)
    runs, fits, models = load(args.raw, args.battery)


    fig_step_responses(runs, args.out)
    fig_step_fit(runs, fits, models, args.pwm, args.out)
    fig_omega_vs_U(fits, models, args.battery, args.out)
    fig_Tm(fits, args.out)
    fig_fit_grid(runs, fits, args.out)
    fig_model_vs_measurement(runs, models, args.out)
    fig_mismatch(fits, args.out)
    fig_motor_graphs(fits, models, args.battery, args.out)
    fig_omega_all_wheels(fits, args.battery, args.out)


if __name__ == "__main__":
    main()

saved figures/01_step_responses.png
saved figures/02_step_response_fit.png
saved figures/03_omega_vs_U.png
saved figures/04_Tm_vs_pwm.png
saved figures/05_fit_grid_wheel_1.png
saved figures/05_fit_grid_wheel_2.png
saved figures/06_model_vs_measurement.png
saved figures/07_wheel_mismatch.png
saved figures/08_motor_graphs.png
saved figures/09_omega_all_wheels.png
